
# Solar Filament Segmentation Challenge 2026 — YOLO26 Segmentation Baseline

This notebook keeps the original baseline structure while using **Ultralytics YOLO26s instance segmentation**.

### Pipeline

- leakage-safe image-level train/validation split
- COCO annotations → YOLO segmentation labels
- 512×512 training/inference size
- early stopping
- validation confidence-threshold selection
- competition-oriented validation metrics
- connected-component cleanup
- native-resolution RLE submission
- training graphs
- validation qualitative review
- unseen test-image qualitative review

### Validation metrics

The validation stage reports:

- **Panoptic Quality (PQ)**
- **Segmentation Quality (SQ)**
- **Recognition Quality (RQ)**
- **Mean Dice**
- **Mean IoU**
- **Mean Precision / Recall**
- **TP / FP / FN instances**
- **One-to-many fragmentation**
- **Many-to-one over-merging**

YOLO's own mask mAP values are retained as model diagnostics, but the custom validation threshold is selected by **PQ** rather than YOLO mAP.

> Test-set ground truth is not available to this notebook, so no test accuracy is calculated.



## 1. Install dependencies and import libraries

YOLO26 segmentation checkpoints use the `-seg` suffix, such as `yolo26s-seg.pt`.


In [ ]:

%pip install -q -U ultralytics pycocotools scikit-learn pandas matplotlib tqdm torchmetrics


In [ ]:

from pathlib import Path
import os
import random
import shutil
import time
import warnings
import multiprocessing
from functools import partial
from concurrent.futures import ThreadPoolExecutor

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm
from torchmetrics.segmentation import DiceScore
from ultralytics import YOLO

warnings.filterwarnings("ignore", category=UserWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
YOLO_DEVICE = 0 if DEVICE.type == "cuda" else "cpu"

print(f"Training device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA devices available: {torch.cuda.device_count()}")



## 2. Configure paths and experiment

GPU 0 is used for this baseline even when multiple GPUs are visible.

The dataset builder uses symlinks where possible, so duplicate COCO records for the same physical JPEG do not require multiple full image copies.


In [ ]:

SEED = 42

IMAGE_SIZE = 2048
BATCH_SIZE = 8 if DEVICE.type == "cuda" else 2
NUM_EPOCHS = 1
VALIDATION_FRACTION = 0.20
PATIENCE = 10

NUM_WORKERS = 4
MIN_COMPONENT_AREA = 20

YOLO_MODEL_NAME = "yolo26s-seg.pt"
YOLO_DATA_DIR = Path.cwd() / "yolo_filament_dataset"
RUNS_DIR = Path.cwd() / "yolo_runs"
RUN_NAME = "yolo26s_filament"

DATA_YAML = YOLO_DATA_DIR / "filament.yaml"
SUBMISSION_PATH = Path.cwd() / "solar_filament_yolo26s_submission.csv"
FINAL_METRICS_PATH = Path.cwd() / "solar_filament_yolo26s_validation_metrics.csv"
PER_IMAGE_METRICS_PATH = Path.cwd() / "solar_filament_yolo26s_per_image_metrics.csv"

REBUILD_YOLO_DATASET = True
CREATE_SUBMISSION = True

CONFIDENCE_CANDIDATES = (0.15, 0.25, 0.35, 0.45, 0.55)
PQ_MATCH_IOU = 0.50


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()


def find_dataset_root():
    expected = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
    candidates = [Path.cwd(), Path.cwd().parent, Path("/kaggle/input")]

    for parent in candidates:
        if not parent.exists():
            continue

        for annotation in parent.glob("**/" + expected):
            root = annotation.parent.parent

            if (
                (root / "train" / "train_images").is_dir()
                and (root / "test" / "test_images").is_dir()
            ):
                return root

    raise FileNotFoundError(
        "Could not find MAGFiLO_1.0_Kaggle_2026 with train and test image folders."
    )


DATA_ROOT = find_dataset_root()
TRAIN_DIR = DATA_ROOT / "train"
TRAIN_IMAGE_DIR = TRAIN_DIR / "train_images"
TEST_IMAGE_DIR = DATA_ROOT / "test" / "test_images"
ANNOTATION_PATH = TRAIN_DIR / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

print(f"Dataset root: {DATA_ROOT}")
print(f"Annotation JSON: {ANNOTATION_PATH}")
print(f"YOLO model: {YOLO_MODEL_NAME}")
print(f"Image size: {IMAGE_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Validation fraction: {VALIDATION_FRACTION:.0%}")
print(f"PQ matching IoU threshold: {PQ_MATCH_IOU:.2f}")



## 3. Load annotations and create a leakage-safe split

The split is performed by **physical filename**, not COCO record ID. Multiple annotation records referring to the same JPEG therefore remain entirely on one side of the split.


In [ ]:

coco = COCO(str(ANNOTATION_PATH))

image_ids = list(coco.imgs.keys())

records = pd.DataFrame([
    {
        "image_id": image_id,
        "file_name": coco.imgs[image_id]["file_name"],
    }
    for image_id in image_ids
])

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=VALIDATION_FRACTION,
    random_state=SEED,
)

train_idx, val_idx = next(
    splitter.split(records, groups=records["file_name"])
)

train_records = records.iloc[train_idx].reset_index(drop=True)
val_records = records.iloc[val_idx].reset_index(drop=True)

train_ids = train_records["image_id"].tolist()
val_ids = val_records["image_id"].tolist()

train_files = set(train_records["file_name"])
val_files = set(val_records["file_name"])

assert train_files.isdisjoint(val_files), "A physical image appears in both splits."

print(f"Training records: {len(train_records)} across {len(train_files)} physical images")
print(f"Validation records: {len(val_records)} across {len(val_files)} physical images")
print(f"Train/validation filename overlap: {len(train_files & val_files)}")



## 4. Convert COCO segmentation masks to YOLO segmentation labels

Ultralytics segmentation labels use one line per instance:

`class_id x1 y1 x2 y2 ...`

with normalized polygon coordinates.

Each COCO record receives a unique sample name. The source JPEG is symlinked when possible and copied otherwise.

The source data remains **grayscale**. Ultralytics receives the grayscale image replicated across three input channels so pretrained RGB weights can be used without adding color information.


In [ ]:

def mask_to_yolo_segments(mask, min_area):
    mask = (mask > 0).astype(np.uint8)
    height, width = mask.shape

    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask,
        connectivity=8,
    )

    segments = []

    for label in range(1, count):
        area = int(stats[label, cv2.CC_STAT_AREA])
        if area < min_area:
            continue

        component = (labels == label).astype(np.uint8)

        contours, _ = cv2.findContours(
            component,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE,
        )

        if not contours:
            continue

        contour = max(contours, key=cv2.contourArea)

        if len(contour) < 3 or cv2.contourArea(contour) <= 0:
            continue

        points = contour.reshape(-1, 2).astype(np.float32)
        points[:, 0] /= width
        points[:, 1] /= height
        points = np.clip(points, 0.0, 1.0)

        if len(points) >= 3:
            segments.append(points)

    return segments


def worker_process_record(
    row_dict,
    annotation_path_str,
    min_area,
    train_image_dir_str,
    image_dir_str,
    label_dir_str,
):
    global _local_coco

    if "_local_coco" not in globals():
        import sys

        old_stdout = sys.stdout
        sys.stdout = open(os.devnull, "w")
        _local_coco = COCO(annotation_path_str)
        sys.stdout = old_stdout

    train_img_dir = Path(train_image_dir_str)
    img_out_dir = Path(image_dir_str)
    lbl_out_dir = Path(label_dir_str)

    image_id = row_dict["image_id"]
    file_name = row_dict["file_name"]

    source_path = train_img_dir / file_name
    if not source_path.is_file():
        return 0

    source_stem = Path(file_name).stem
    sample_stem = f"{image_id}_{source_stem}"

    image_path = img_out_dir / f"{sample_stem}.jpg"
    label_path = lbl_out_dir / f"{sample_stem}.txt"

    image_path.parent.mkdir(parents=True, exist_ok=True)

    if image_path.exists() or image_path.is_symlink():
        image_path.unlink()

    try:
        image_path.symlink_to(source_path.resolve())
    except OSError:
        shutil.copy2(source_path, image_path)

    annotations = _local_coco.loadAnns(
        _local_coco.getAnnIds(imgIds=[image_id])
    )

    lines = []

    for annotation in annotations:
        mask = _local_coco.annToMask(annotation).astype(np.uint8)

        for points in mask_to_yolo_segments(mask, min_area):
            flat = points.reshape(-1)
            values = ["0"]
            values.extend(f"{value:.6f}" for value in flat)
            lines.append(" ".join(values))

    label_path.write_text(
        "\n".join(lines) + ("\n" if lines else ""),
        encoding="utf-8",
    )

    return len(lines)


def write_yolo_split_parallel(split_records, split_name):
    print(f"\n[START] Building YOLO split: '{split_name}'", flush=True)

    image_dir = YOLO_DATA_DIR / "images" / split_name
    label_dir = YOLO_DATA_DIR / "labels" / split_name

    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    records_list = split_records[
        ["image_id", "file_name"]
    ].to_dict(orient="records")

    worker_task = partial(
        worker_process_record,
        annotation_path_str=str(ANNOTATION_PATH),
        min_area=MIN_COMPONENT_AREA,
        train_image_dir_str=str(TRAIN_IMAGE_DIR),
        image_dir_str=str(image_dir),
        label_dir_str=str(label_dir),
    )

    empty_labels = 0
    total_instances = 0

    with multiprocessing.Pool(processes=NUM_WORKERS) as pool:
        results = tqdm(
            pool.imap(worker_task, records_list, chunksize=10),
            total=len(records_list),
            desc=f"Preparing YOLO {split_name}",
        )

        for idx, instance_count in enumerate(results, start=1):
            total_instances += instance_count

            if instance_count == 0:
                empty_labels += 1

            if idx == 1 or idx % 100 == 0 or idx == len(records_list):
                print(
                    f"[{split_name.upper()}] {idx}/{len(records_list)} processed | "
                    f"polygons: {total_instances:,} | "
                    f"empty labels: {empty_labels}",
                    flush=True,
                )

    print(
        f"[FINISHED] {split_name}: {len(records_list)} records | "
        f"{total_instances:,} polygon instances | "
        f"empty labels: {empty_labels}\n",
        flush=True,
    )


if REBUILD_YOLO_DATASET and YOLO_DATA_DIR.exists():
    print(f"[CLEANUP] Removing old directory: {YOLO_DATA_DIR}", flush=True)
    shutil.rmtree(YOLO_DATA_DIR)

write_yolo_split_parallel(train_records, "train")
write_yolo_split_parallel(val_records, "val")

DATA_YAML.write_text(
    (
        f"path: '{YOLO_DATA_DIR.as_posix()}'\n"
        "train: images/train\n"
        "val: images/val\n"
        "names:\n"
        "  0: filament\n"
    ),
    encoding="utf-8",
)

print(f"YOLO dataset configuration: {DATA_YAML}")
print(DATA_YAML.read_text())



## 5. Build and train YOLO26s-seg

Augmentation is kept relatively close to the original U-Net baseline:

- horizontal/vertical flips
- small rotations
- small translation and scaling
- mild value variation
- no mosaic/mixup/copy-paste

`NUM_EPOCHS = 1` is intentionally retained from the supplied baseline as a smoke test. Increase it for an actual experiment.


In [ ]:

print("Building pretrained YOLO26s segmentation model...")

model = YOLO(YOLO_MODEL_NAME)

training_start = time.perf_counter()

train_results = model.train(
    data=str(DATA_YAML),
    epochs=NUM_EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=YOLO_DEVICE,
    workers=NUM_WORKERS,
    patience=PATIENCE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    optimizer="auto",
    amp=(DEVICE.type == "cuda"),
    seed=SEED,
    deterministic=True,
    fliplr=0.5,
    flipud=0.5,
    degrees=12.0,
    translate=0.04,
    scale=0.08,
    shear=0.0,
    perspective=0.0,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.15,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    cache="disk",
    plots=False,
    verbose=True,
)

training_time = time.perf_counter() - training_start

try:
    run_dir = Path(model.trainer.save_dir)
except AttributeError:
    run_dir = RUNS_DIR / RUN_NAME

best_checkpoint = run_dir / "weights" / "best.pt"

if not best_checkpoint.is_file():
    raise FileNotFoundError(
        f"Best checkpoint was not created: {best_checkpoint}"
    )

model = YOLO(str(best_checkpoint))

print(
    f"\n[SUCCESS] Training complete!\n"
    f"Run directory: {run_dir}\n"
    f"Best checkpoint: {best_checkpoint}\n"
    f"Total time elapsed: {training_time / 60:.1f} min",
    flush=True,
)



## 6. Read training history

Ultralytics writes `results.csv` in the run directory. These plots show the model's training/validation loss and its built-in mask mAP diagnostics.


In [ ]:

history_path = run_dir / "results.csv"

if not history_path.is_file():
    raise FileNotFoundError(f"YOLO training history not found: {history_path}")

history = pd.read_csv(history_path)
history.columns = [column.strip() for column in history.columns]

print("Available history columns:")
print(", ".join(history.columns))

epochs = np.arange(1, len(history) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for column, label in [
    ("train/seg_loss", "Train segmentation loss"),
    ("val/seg_loss", "Validation segmentation loss"),
]:
    if column in history.columns:
        axes[0].plot(
            epochs,
            history[column],
            marker="o",
            label=label,
        )

axes[0].set(
    title="YOLO segmentation loss",
    xlabel="Epoch",
    ylabel="Loss",
)
axes[0].legend()
axes[0].grid(alpha=0.25)

for column, label in [
    ("metrics/mAP50(M)", "Mask mAP50"),
    ("metrics/mAP50-95(M)", "Mask mAP50-95"),
]:
    if column in history.columns:
        axes[1].plot(
            epochs,
            history[column],
            marker="o",
            label=label,
        )

axes[1].set(
    title="YOLO validation mask metrics",
    xlabel="Epoch",
    ylabel="Metric",
    ylim=(0, 1),
)
axes[1].legend()
axes[1].grid(alpha=0.25)

fig.tight_layout()
plt.show()

print(f"Best checkpoint: {best_checkpoint}")



## 7. Prediction, metric, and submission helpers

The validation and test pipelines use the same prediction path:

1. run YOLO instance segmentation
2. apply the confidence threshold
3. union the selected masks
4. split connected components
5. evaluate/encode the resulting filament instances


In [ ]:

def to_yolo_image(image):
    if image.ndim != 2:
        raise ValueError(f"Expected grayscale image, got shape {image.shape}")

    return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)


def predict_instances(image, min_confidence=0.15):
    source = to_yolo_image(image)

    results = model.predict(
        source=source,
        imgsz=IMAGE_SIZE,
        conf=min_confidence,
        iou=0.7,
        retina_masks=True,
        device=YOLO_DEVICE,
        verbose=False,
    )

    result = results[0]

    if result.masks is None or result.boxes is None:
        return []

    masks = result.masks.data.detach().float().cpu().numpy()
    confidences = result.boxes.conf.detach().float().cpu().numpy()

    native_height, native_width = image.shape
    instances = []

    for mask, confidence in zip(masks, confidences):
        if mask.shape != image.shape:
            mask = cv2.resize(
                mask,
                (native_width, native_height),
                interpolation=cv2.INTER_NEAREST,
            )

        instances.append(
            {
                "mask": mask >= 0.5,
                "confidence": float(confidence),
            }
        )

    return instances


def union_instances(instances, confidence_threshold):
    selected = [
        instance
        for instance in instances
        if instance["confidence"] >= confidence_threshold
    ]

    if not selected:
        return None

    foreground = np.zeros_like(
        selected[0]["mask"],
        dtype=bool,
    )

    for instance in selected:
        foreground |= instance["mask"]

    return foreground


def split_into_filaments(binary_mask, min_area=MIN_COMPONENT_AREA):
    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary_mask.astype(np.uint8),
        connectivity=8,
    )

    components = [
        labels == label
        for label in range(1, count)
        if stats[label, cv2.CC_STAT_AREA] >= min_area
    ]

    def component_key(component):
        ys, xs = np.where(component)
        return (
            -int(component.sum()),
            int(ys.min()) if len(ys) else 0,
            int(xs.min()) if len(xs) else 0,
        )

    return sorted(components, key=component_key)


def build_final_prediction_instances(instances, confidence_threshold):
    combined = union_instances(
        instances,
        confidence_threshold,
    )

    if combined is None:
        return []

    return split_into_filaments(
        combined,
        MIN_COMPONENT_AREA,
    )


def mask_to_rle(binary_mask):
    encoded = mask_utils.encode(
        np.asfortranarray(
            binary_mask.astype(np.uint8)
        )
    )
    return encoded["counts"].decode("utf-8")


def get_test_image_paths():
    paths = [
        path
        for path in TEST_IMAGE_DIR.iterdir()
        if path.is_file()
        and path.suffix.lower() in {".jpg", ".jpeg"}
    ]

    return sorted(paths, key=lambda path: path.name)



## 8. Competition-oriented validation

The held-out validation set is evaluated for each confidence threshold.

### Pixel metrics

- Mean Dice — calculated **per image** and then averaged
- Mean IoU — calculated **per image** and then averaged
- Mean precision / recall

### Instance metrics

- PQ
- SQ
- RQ
- TP / FP / FN
- one-to-many relations
- many-to-one relations

Threshold selection uses **mean PQ** because it is the primary competition metric in the supplied competition specification.


In [ ]:

dice_metric = DiceScore(
    num_classes=2,
    include_background=False,
    average="micro",
    input_format="index",
)


def calculate_pixel_metrics(prediction, target):
    prediction_tensor = torch.from_numpy(
        prediction.astype(np.int64)
    ).unsqueeze(0)

    target_tensor = torch.from_numpy(
        target.astype(np.int64)
    ).unsqueeze(0)

    prediction_has_foreground = bool(prediction.any())
    target_has_foreground = bool(target.any())

    if not prediction_has_foreground and not target_has_foreground:
        dice = 1.0
    else:
        dice = float(
            dice_metric(
                prediction_tensor,
                target_tensor,
            ).item()
        )

        if np.isnan(dice):
            dice = 0.0

    intersection = np.logical_and(
        prediction,
        target,
    ).sum()

    union = np.logical_or(
        prediction,
        target,
    ).sum()

    iou = (
        float(intersection / union)
        if union > 0
        else 1.0
    )

    true_positive = int(intersection)
    false_positive = int(
        np.logical_and(prediction, ~target).sum()
    )
    false_negative = int(
        np.logical_and(~prediction, target).sum()
    )

    precision = (
        true_positive / (true_positive + false_positive)
        if true_positive + false_positive > 0
        else 1.0
    )

    recall = (
        true_positive / (true_positive + false_negative)
        if true_positive + false_negative > 0
        else 1.0
    )

    return {
        "pixel_dice": dice,
        "pixel_iou": iou,
        "pixel_precision": float(precision),
        "pixel_recall": float(recall),
    }


def calculate_instance_metrics(
    prediction_instances,
    target_instances,
    iou_threshold=PQ_MATCH_IOU,
):
    num_predictions = len(prediction_instances)
    num_targets = len(target_instances)

    if num_predictions == 0 and num_targets == 0:
        return {
            "pq": 1.0,
            "sq": 1.0,
            "rq": 1.0,
            "tp_instances": 0,
            "fp_instances": 0,
            "fn_instances": 0,
            "one_to_many": 0,
            "many_to_one": 0,
            "matched_iou_sum": 0.0,
        }

    if num_predictions == 0:
        return {
            "pq": 0.0,
            "sq": 0.0,
            "rq": 0.0,
            "tp_instances": 0,
            "fp_instances": 0,
            "fn_instances": num_targets,
            "one_to_many": 0,
            "many_to_one": 0,
            "matched_iou_sum": 0.0,
        }

    if num_targets == 0:
        return {
            "pq": 0.0,
            "sq": 0.0,
            "rq": 0.0,
            "tp_instances": 0,
            "fp_instances": num_predictions,
            "fn_instances": 0,
            "one_to_many": 0,
            "many_to_one": 0,
            "matched_iou_sum": 0.0,
        }

    iou_matrix = np.zeros(
        (num_targets, num_predictions),
        dtype=np.float32,
    )

    for gt_index, gt_mask in enumerate(target_instances):
        gt_area = int(gt_mask.sum())
        if gt_area == 0:
            continue

        for pred_index, pred_mask in enumerate(prediction_instances):
            pred_area = int(pred_mask.sum())
            if pred_area == 0:
                continue

            intersection = np.logical_and(
                gt_mask,
                pred_mask,
            ).sum()

            if intersection == 0:
                continue

            union = gt_area + pred_area - intersection
            iou_matrix[gt_index, pred_index] = intersection / union

    gt_relation_counts = (
        iou_matrix > iou_threshold
    ).sum(axis=1)
    pred_relation_counts = (
        iou_matrix > iou_threshold
    ).sum(axis=0)

    one_to_many = int(
        (gt_relation_counts > 1).sum()
    )
    many_to_one = int(
        (pred_relation_counts > 1).sum()
    )

    candidate_pairs = []

    for gt_index in range(num_targets):
        for pred_index in range(num_predictions):
            iou = float(
                iou_matrix[gt_index, pred_index]
            )

            if iou > iou_threshold:
                candidate_pairs.append(
                    (iou, gt_index, pred_index)
                )

    candidate_pairs.sort(
        key=lambda item: item[0],
        reverse=True,
    )

    matched_gt = set()
    matched_predictions = set()
    matched_ious = []

    for iou, gt_index, pred_index in candidate_pairs:
        if gt_index in matched_gt:
            continue
        if pred_index in matched_predictions:
            continue

        matched_gt.add(gt_index)
        matched_predictions.add(pred_index)
        matched_ious.append(iou)

    tp = len(matched_ious)
    fp = num_predictions - tp
    fn = num_targets - tp
    matched_iou_sum = float(np.sum(matched_ious))

    denominator = tp + 0.5 * fp + 0.5 * fn

    pq = (
        matched_iou_sum / denominator
        if denominator > 0
        else 0.0
    )

    sq = (
        matched_iou_sum / tp
        if tp > 0
        else 0.0
    )

    rq = (
        tp / denominator
        if denominator > 0
        else 0.0
    )

    return {
        "pq": float(pq),
        "sq": float(sq),
        "rq": float(rq),
        "tp_instances": tp,
        "fp_instances": fp,
        "fn_instances": fn,
        "one_to_many": one_to_many,
        "many_to_one": many_to_one,
        "matched_iou_sum": matched_iou_sum,
    }


In [ ]:

def _worker_load_val_data(
    row_dict,
    annotation_path_str,
    train_image_dir_str,
):
    global _local_val_coco

    if "_local_val_coco" not in globals():
        import sys

        old_stdout = sys.stdout
        sys.stdout = open(os.devnull, "w")
        _local_val_coco = COCO(annotation_path_str)
        sys.stdout = old_stdout

    image_id = row_dict["image_id"]
    file_name = row_dict["file_name"]

    image_path = Path(train_image_dir_str) / file_name
    image = cv2.imread(
        str(image_path),
        cv2.IMREAD_GRAYSCALE,
    )

    if image is None:
        return None

    target_instances = []
    annotations = _local_val_coco.loadAnns(
        _local_val_coco.getAnnIds(imgIds=[image_id])
    )

    for annotation in annotations:
        mask = _local_val_coco.annToMask(annotation).astype(bool)
        if mask.any():
            target_instances.append(mask)

    target_union = np.zeros(
        image.shape,
        dtype=bool,
    )

    for mask in target_instances:
        target_union |= mask

    return {
        "image_id": image_id,
        "file_name": file_name,
        "image": image,
        "target_instances": target_instances,
        "target_union": target_union,
    }


validation_start = time.perf_counter()
minimum_confidence = min(CONFIDENCE_CANDIDATES)
validation_examples = []

threshold_results = {
    threshold: {
        "per_image": []
    }
    for threshold in CONFIDENCE_CANDIDATES
}

val_records_list = val_records[
    ["image_id", "file_name"]
].to_dict(orient="records")

print(
    f"[START] Validation on {len(val_records_list):,} records",
    flush=True,
)

worker_func = partial(
    _worker_load_val_data,
    annotation_path_str=str(ANNOTATION_PATH),
    train_image_dir_str=str(TRAIN_IMAGE_DIR),
)

with multiprocessing.Pool(processes=NUM_WORKERS) as pool:
    data_iterator = pool.imap(
        worker_func,
        val_records_list,
        chunksize=5,
    )

    progress = tqdm(
        data_iterator,
        total=len(val_records_list),
        desc="Validation inference",
        unit="record",
    )

    for idx, sample_data in enumerate(progress, start=1):
        if sample_data is None:
            continue

        image = sample_data["image"]
        target_instances = sample_data["target_instances"]
        target_union = sample_data["target_union"]
        file_name = sample_data["file_name"]

        instances = predict_instances(
            image,
            min_confidence=minimum_confidence,
        )

        for threshold in CONFIDENCE_CANDIDATES:
            prediction_instances = build_final_prediction_instances(
                instances,
                threshold,
            )

            prediction_union = np.zeros_like(
                target_union,
                dtype=bool,
            )

            for prediction_instance in prediction_instances:
                prediction_union |= prediction_instance

            pixel_metrics = calculate_pixel_metrics(
                prediction_union,
                target_union,
            )

            instance_metrics = calculate_instance_metrics(
                prediction_instances,
                target_instances,
            )

            threshold_results[threshold]["per_image"].append(
                {
                    "file_name": file_name,
                    "pixel_dice": pixel_metrics["pixel_dice"],
                    "pixel_iou": pixel_metrics["pixel_iou"],
                    "pixel_precision": pixel_metrics["pixel_precision"],
                    "pixel_recall": pixel_metrics["pixel_recall"],
                    "pq": instance_metrics["pq"],
                    "sq": instance_metrics["sq"],
                    "rq": instance_metrics["rq"],
                    "tp_instances": instance_metrics["tp_instances"],
                    "fp_instances": instance_metrics["fp_instances"],
                    "fn_instances": instance_metrics["fn_instances"],
                    "predicted_instances": len(prediction_instances),
                    "ground_truth_instances": len(target_instances),
                    "one_to_many": instance_metrics["one_to_many"],
                    "many_to_one": instance_metrics["many_to_one"],
                }
            )

        if len(validation_examples) < 4:
            validation_examples.append(
                {
                    "image": image,
                    "target": target_union,
                    "instances": instances,
                    "file_name": file_name,
                }
            )

        if idx == 1 or idx % 50 == 0 or idx == len(val_records_list):
            print(
                f"[VAL PROGRESS] Evaluated {idx}/{len(val_records_list)} records",
                flush=True,
            )


threshold_rows = []

for threshold in CONFIDENCE_CANDIDATES:
    per_image_df = pd.DataFrame(
        threshold_results[threshold]["per_image"]
    )

    threshold_rows.append(
        {
            "confidence": threshold,
            "PQ": per_image_df["pq"].mean(),
            "SQ": per_image_df["sq"].mean(),
            "RQ": per_image_df["rq"].mean(),
            "Mean Dice": per_image_df["pixel_dice"].mean(),
            "Mean IoU": per_image_df["pixel_iou"].mean(),
            "Mean Precision": per_image_df["pixel_precision"].mean(),
            "Mean Recall": per_image_df["pixel_recall"].mean(),
            "TP": int(per_image_df["tp_instances"].sum()),
            "FP": int(per_image_df["fp_instances"].sum()),
            "FN": int(per_image_df["fn_instances"].sum()),
            "Predicted Instances": int(per_image_df["predicted_instances"].sum()),
            "GT Instances": int(per_image_df["ground_truth_instances"].sum()),
            "One-to-Many": int(per_image_df["one_to_many"].sum()),
            "Many-to-One": int(per_image_df["many_to_one"].sum()),
        }
    )

threshold_summary = (
    pd.DataFrame(threshold_rows)
    .sort_values("PQ", ascending=False)
    .reset_index(drop=True)
)

BEST_CONFIDENCE = float(
    threshold_summary.iloc[0]["confidence"]
)

validation_time = time.perf_counter() - validation_start

print("\n" + "=" * 90)
print("                         VALIDATION RESULTS")
print("=" * 90)

display(
    threshold_summary.round(4)
)

print(
    f"\nSelected confidence threshold: {BEST_CONFIDENCE:.2f}"
)
print(
    f"Best validation PQ: {threshold_summary.iloc[0]['PQ']:.4f}"
)
print(
    f"Validation inference time: {validation_time:.1f}s"
)



## 9. Generate the competition submission

The final test pipeline uses the exact same instance selection and connected-component cleanup as validation, then encodes each final filament component at native resolution using COCO-style RLE.


In [ ]:

def _parallel_process_single_canvas(args):
    combined_mask, min_area, path_stem = args
    rows = []

    components = split_into_filaments(
        combined_mask,
        min_area,
    )

    for filament_counter, component in enumerate(
        components,
        start=1,
    ):
        rows.append(
            {
                "filament_id": f"{path_stem}_{filament_counter}",
                "segmentation_rle": mask_to_rle(component),
            }
        )

    return rows


def predict_combined_canvas(image, min_confidence):
    instances = predict_instances(
        image,
        min_confidence=min_confidence,
    )

    combined = union_instances(
        instances,
        min_confidence,
    )

    if combined is None:
        return np.zeros(
            image.shape,
            dtype=bool,
        )

    return combined


def create_submission_parallel(output_path=SUBMISSION_PATH):
    paths = get_test_image_paths()

    if not paths:
        raise FileNotFoundError(
            f"No .jpg/.jpeg images found in {TEST_IMAGE_DIR}"
        )

    total_images = len(paths)
    start_time = time.perf_counter()

    print(
        f"[START] Generating predictions for {total_images:,} test images "
        f"using threshold {BEST_CONFIDENCE:.2f}",
        flush=True,
    )

    tasks = []

    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        def load_img(path):
            return path, cv2.imread(
                str(path),
                cv2.IMREAD_GRAYSCALE,
            )

        progress = tqdm(
            executor.map(load_img, paths),
            total=total_images,
            desc="Submission inference",
            unit="img",
        )

        for image_index, (path, image) in enumerate(
            progress,
            start=1,
        ):
            if image is None:
                raise OSError(
                    f"Could not read test image: {path}"
                )

            combined_mask = predict_combined_canvas(
                image,
                min_confidence=BEST_CONFIDENCE,
            )

            tasks.append(
                (
                    combined_mask,
                    MIN_COMPONENT_AREA,
                    path.stem,
                )
            )

            if image_index == 1 or image_index % 50 == 0 or image_index == total_images:
                print(
                    f"[INFERENCE PROGRESS] {image_index}/{total_images} images",
                    flush=True,
                )

    print(
        f"[PARALLEL] Encoding {len(tasks):,} predicted images on CPU",
        flush=True,
    )

    rows = []

    with multiprocessing.Pool(processes=NUM_WORKERS) as pool:
        worker_results = pool.map(
            _parallel_process_single_canvas,
            tasks,
        )

    for encoded_rows in worker_results:
        rows.extend(encoded_rows)

    submission = pd.DataFrame(
        rows,
        columns=[
            "filament_id",
            "segmentation_rle",
        ],
    )

    if not submission.empty:
        if submission["filament_id"].duplicated().any():
            raise ValueError(
                "Generated submission contains duplicate filament IDs."
            )

    submission.to_csv(
        output_path,
        index=False,
    )

    elapsed = time.perf_counter() - start_time

    print(
        f"\n[SUCCESS] Submission complete!\n"
        f"Processed {total_images:,} images | "
        f"Extracted {len(submission):,} filament rows\n"
        f"Total runtime: {elapsed / 60:.2f} min\n"
        f"Saved to: {output_path}",
        flush=True,
    )

    return submission


if CREATE_SUBMISSION:
    submission_df = create_submission_parallel()
    display(submission_df.head())
else:
    submission_df = None
    print("CREATE_SUBMISSION=False; submission generation skipped.")



## 10. Submission audit and final validation report

This section runs after submission creation and gives one final competition-oriented summary.

The validation values below refer to the **same confidence threshold used for the submission**.


In [ ]:

if CREATE_SUBMISSION:
    audit_df = pd.read_csv(SUBMISSION_PATH)
else:
    audit_df = pd.DataFrame(
        columns=[
            "filament_id",
            "segmentation_rle",
        ]
    )

print("--- Submission File Integrity Audit ---")
print(f"Total rows: {len(audit_df):,}")
print(f"Null values: {audit_df.isnull().sum().sum():,}")

if "filament_id" in audit_df.columns:
    print(
        f"Duplicate filament IDs: "
        f"{audit_df['filament_id'].duplicated().sum():,}"
    )

if "segmentation_rle" in audit_df.columns:
    empty_rle_count = (
        audit_df["segmentation_rle"]
        .fillna("")
        .astype(str)
        .str.len()
        .eq(0)
        .sum()
    )
else:
    empty_rle_count = 0

print(f"Empty RLE strings: {empty_rle_count:,}")

if len(audit_df) > 0:
    sample_rle = audit_df.iloc[0]["segmentation_rle"]
    is_string_valid = (
        isinstance(sample_rle, str)
        and len(sample_rle) > 0
    )
else:
    is_string_valid = False

print(f"Sample RLE structural check: {is_string_valid}")

submission_valid = (
    set(audit_df.columns) == {
        "filament_id",
        "segmentation_rle",
    }
    and audit_df.isnull().sum().sum() == 0
    and audit_df["filament_id"].duplicated().sum() == 0
    and empty_rle_count == 0
    and is_string_valid
)

if submission_valid:
    print("\n[VERIFIED] Submission structure passed the local audit.")
else:
    print("\n[WARNING] Submission structure needs inspection.")


best_row = threshold_summary[
    threshold_summary["confidence"] == BEST_CONFIDENCE
].iloc[0]

final_metrics = pd.DataFrame(
    {
        "Metric": [
            "Panoptic Quality (PQ)",
            "Segmentation Quality (SQ)",
            "Recognition Quality (RQ)",
            "Mean Dice",
            "Mean IoU",
            "Mean Precision",
            "Mean Recall",
            "True Positives",
            "False Positives",
            "False Negatives",
            "Predicted Instances",
            "Ground-Truth Instances",
            "One-to-Many (Fragmentation)",
            "Many-to-One (Over-Merging)",
            "Selected Confidence",
        ],
        "Value": [
            best_row["PQ"],
            best_row["SQ"],
            best_row["RQ"],
            best_row["Mean Dice"],
            best_row["Mean IoU"],
            best_row["Mean Precision"],
            best_row["Mean Recall"],
            int(best_row["TP"]),
            int(best_row["FP"]),
            int(best_row["FN"]),
            int(best_row["Predicted Instances"]),
            int(best_row["GT Instances"]),
            int(best_row["One-to-Many"]),
            int(best_row["Many-to-One"]),
            BEST_CONFIDENCE,
        ],
    }
)

print("\n" + "=" * 75)
print("                     FINAL VALIDATION REPORT")
print("=" * 75)
display(final_metrics.round(4))

selected_per_image = pd.DataFrame(
    threshold_results[BEST_CONFIDENCE]["per_image"]
)

print("\n--- Per-image metric distribution ---")

distribution = (
    selected_per_image[
        [
            "pixel_dice",
            "pixel_iou",
            "pq",
            "sq",
            "rq",
        ]
    ]
    .agg(
        [
            "mean",
            "median",
            "std",
            "min",
            "max",
        ]
    )
    .T
)

display(distribution.round(4))

best_mask_map50 = (
    float(history["metrics/mAP50(M)"].max())
    if "metrics/mAP50(M)" in history.columns
    else np.nan
)

best_mask_map50_95 = (
    float(history["metrics/mAP50-95(M)"].max())
    if "metrics/mAP50-95(M)" in history.columns
    else np.nan
)

diagnostics = pd.DataFrame(
    {
        "Diagnostic": [
            "YOLO best mask mAP50",
            "YOLO best mask mAP50-95",
            "Validation records",
            "Submission rows",
            "Submission valid",
        ],
        "Value": [
            best_mask_map50,
            best_mask_map50_95,
            len(selected_per_image),
            len(audit_df),
            submission_valid,
        ],
    }
)

print("\n--- Model / submission diagnostics ---")
display(diagnostics.round(4))

final_metrics.to_csv(
    FINAL_METRICS_PATH,
    index=False,
)

selected_per_image.to_csv(
    PER_IMAGE_METRICS_PATH,
    index=False,
)

print(f"Saved validation summary: {FINAL_METRICS_PATH}")
print(f"Saved per-image metrics: {PER_IMAGE_METRICS_PATH}")



## 11. Confidence-threshold visualization and validation examples

The threshold graph uses **PQ**, matching the threshold-selection criterion.


In [ ]:

fig, axis = plt.subplots(figsize=(7, 4.5))

axis.plot(
    threshold_summary["confidence"],
    threshold_summary["PQ"],
    marker="o",
    label="Validation PQ",
)

axis.axvline(
    BEST_CONFIDENCE,
    linestyle="--",
    label=f"Selected: {BEST_CONFIDENCE:.2f}",
)

axis.set(
    title="Validation PQ vs confidence threshold",
    xlabel="YOLO instance confidence",
    ylabel="Panoptic Quality (PQ)",
    ylim=(0, 1),
)

axis.grid(alpha=0.25)
axis.legend()

fig.tight_layout()
plt.show()


for example in validation_examples:
    image = example["image"]
    target = example["target"]
    instances = example["instances"]

    prediction = union_instances(
        instances,
        BEST_CONFIDENCE,
    )

    if prediction is None:
        prediction = np.zeros_like(
            target,
            dtype=bool,
        )

    selected_prediction_instances = build_final_prediction_instances(
        instances,
        BEST_CONFIDENCE,
    )

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(14, 4.5),
    )

    axes[0].imshow(
        image,
        cmap="gray",
    )
    axes[0].set_title("Input image")

    axes[1].imshow(
        target,
        cmap="gray",
        vmin=0,
        vmax=1,
    )
    axes[1].set_title("Ground truth")

    axes[2].imshow(
        prediction,
        cmap="gray",
        vmin=0,
        vmax=1,
    )
    axes[2].set_title(
        f"Prediction · "
        f"{len(selected_prediction_instances)} instance(s)"
    )

    for axis in axes:
        axis.axis("off")

    fig.suptitle(example["file_name"])
    fig.tight_layout()
    plt.show()



## 12. Display a labeled prediction on an unseen test image

The test image is used only for inference and visualization. No test ground truth is used.


In [ ]:

def plot_labeled_prediction(image, instances, title):
    selected_instances = [
        instance
        for instance in instances
        if instance["confidence"] >= BEST_CONFIDENCE
    ]

    combined = union_instances(
        instances,
        BEST_CONFIDENCE,
    )

    components = (
        split_into_filaments(combined)
        if combined is not None
        else []
    )

    fig, axis = plt.subplots(figsize=(9, 9))
    axis.imshow(image, cmap="gray")

    for filament_index, component in enumerate(
        components,
        start=1,
    ):
        axis.contour(
            component.astype(np.uint8),
            levels=[0.5],
            linewidths=1.2,
        )

        moments = cv2.moments(
            component.astype(np.uint8)
        )

        if moments["m00"]:
            center_x = moments["m10"] / moments["m00"]
            center_y = moments["m01"] / moments["m00"]
        else:
            ys, xs = np.where(component)
            center_x = xs.mean()
            center_y = ys.mean()

        area = int(component.sum())
        axis.text(
            center_x,
            center_y,
            f"F{filament_index}\n{area:,} px",
            fontsize=8,
            ha="center",
            va="center",
            bbox=dict(
                boxstyle="round,pad=0.2",
                facecolor="white",
                alpha=0.75,
            ),
        )

    axis.set_title(
        f"{title} · {len(components)} predicted filament(s)"
    )
    axis.axis("off")

    fig.tight_layout()
    plt.show()


test_paths = get_test_image_paths()

test_path = test_paths[0]
test_image = cv2.imread(
    str(test_path),
    cv2.IMREAD_GRAYSCALE,
)

if test_image is None:
    raise OSError(
        f"Could not read test image: {test_path}"
    )

test_instances = predict_instances(
    test_image,
    min_confidence=BEST_CONFIDENCE,
)

print(
    f"Displaying labeled prediction for test image: {test_path.name}"
)

plot_labeled_prediction(
    test_image,
    test_instances,
    f"Test prediction · {test_path.name} · confidence {BEST_CONFIDENCE:.2f}",
)



## 13. Experiment summary

### Main outputs

- `yolo_filament_dataset/` — temporary YOLO-format training dataset
- `yolo_runs/yolo26s_filament/weights/best.pt` — best YOLO checkpoint
- `solar_filament_yolo26s_submission.csv` — competition submission
- `solar_filament_yolo26s_validation_metrics.csv` — final validation summary
- `solar_filament_yolo26s_per_image_metrics.csv` — per-image validation metrics

### Important interpretation

The validation values are estimates from the held-out validation split. They are not the Kaggle leaderboard score. The test-set ground truth remains unavailable to this notebook.

The primary custom threshold-selection metric is **PQ**, while Mean Dice, Mean IoU, and the structural error counts are reported alongside it.
